# 05 HRV Domain And Robustness

Reviewer-facing HRV-domain and robustness readiness/status notebook.


## Setup


In [ ]:
from pathlib import Path
import json
import os
import subprocess
import sys
import time

# Keep all git operations out of Drive-backed cwd. This avoids Colab
# "Transport endpoint is not connected" failures after Drive hiccups.
os.chdir('/content')
print('cwd reset:', Path.cwd())

try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception as exc:
    print(f'Drive mount skipped or already mounted: {exc}')

REPO_URL = 'https://github.com/BrianNguyen29/ECG-RAMBA.git'
BRANCH = os.environ.get('ECG_RAMBA_BRANCH', 'main')
DRIVE_ROOT = Path('/content/drive/MyDrive/ECG-Ramba')
MIRROR_REVISION_ROOT = DRIVE_ROOT / 'revision_artifacts' / 'reports' / 'revision'
LOCAL_RUNTIME_ROOT = Path('/content/ecg_ramba_runtime')
REPO_DIR = Path('/content/ECG-RAMBA')

os.environ['ECG_RAMBA_DRIVE_ROOT'] = str(DRIVE_ROOT)
os.environ.setdefault('ECG_RAMBA_LOCAL_ROOT', str(LOCAL_RUNTIME_ROOT))
os.environ.setdefault('ECG_RAMBA_EXTRACT_DIR', str(LOCAL_RUNTIME_ROOT / 'chapman'))
os.environ.setdefault('ECG_RAMBA_USE_CLEAN_CACHE', '0')
os.environ.setdefault('ECG_RAMBA_SAVE_CLEAN_CACHE', '0')

def _run_setup(cmd, cwd='/content', check=True):
    print(f'$ {cmd}', flush=True)
    result = subprocess.run(
        cmd,
        shell=True,
        cwd=str(cwd) if cwd else None,
        check=False,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
    )
    if result.stdout:
        print(result.stdout.rstrip())
    if check and result.returncode:
        raise subprocess.CalledProcessError(result.returncode, cmd, output=result.stdout)
    return result

def _is_repo(path):
    path = Path(path)
    return (path / '.git').exists() and (path / 'configs' / 'config.py').exists()

def _remove_local_repo(reason):
    if REPO_DIR.exists():
        print(f'Removing local repo at {REPO_DIR}: {reason}')
        _run_setup(f'rm -rf "{REPO_DIR}"', cwd='/content')

if REPO_DIR.exists() and not _is_repo(REPO_DIR):
    _remove_local_repo('path exists but is not a valid repo')

if not REPO_DIR.exists():
    _run_setup(f'git clone --branch {BRANCH} {REPO_URL} "{REPO_DIR}"', cwd='/content')
else:
    print('Using existing local repo:', REPO_DIR)

lock_path = REPO_DIR / '.git' / 'index.lock'
if lock_path.exists():
    print('Removing stale git index lock:', lock_path)
    lock_path.unlink()

fetch_result = _run_setup('git fetch origin', cwd=REPO_DIR, check=False)
checkout_result = _run_setup(f'git checkout {BRANCH}', cwd=REPO_DIR, check=False)
if fetch_result.returncode == 0 and checkout_result.returncode == 0:
    pull_result = _run_setup(f'git pull --ff-only --autostash origin {BRANCH}', cwd=REPO_DIR, check=False)
    if pull_result.returncode:
        print('git pull failed; replacing /content clone with a fresh checkout.')
        _remove_local_repo('pull failed')
        _run_setup(f'git clone --branch {BRANCH} {REPO_URL} "{REPO_DIR}"', cwd='/content')
else:
    print('git fetch/checkout failed; replacing /content clone with a fresh checkout.')
    _remove_local_repo('fetch or checkout failed')
    _run_setup(f'git clone --branch {BRANCH} {REPO_URL} "{REPO_DIR}"', cwd='/content')

os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

print('cwd       :', Path.cwd())
print('drive_root:', DRIVE_ROOT)
print('repo_dir  :', REPO_DIR)
print('branch    :', BRANCH)
_run_setup('git rev-parse --short HEAD', cwd=REPO_DIR, check=False)
_run_setup('git status --short --branch', cwd=REPO_DIR, check=False)


DRIVE_REPO_REVISION_ROOT = DRIVE_ROOT / 'ECG-RAMBA' / 'reports' / 'revision'
REQUIRED_REVISION_ARTIFACTS_FOR_05 = [
    'manifests/oof_final_ema_freeze_manifest.json',
    'predictions/oof_final_ema_predictions.npz',
    'metrics/calibration_ci_oof_final_ema_predictions.json',
    'metrics/baseline_summary.csv',
    'metrics/component_check_summary.json',
    'audit_protocol.json',
    'a0_resolution_status.json',
    'hrv36_schema.csv',
]


def _sha256_local(path):
    import hashlib
    digest = hashlib.sha256()
    with Path(path).open('rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()


def _mirror_manifest_rows(mirror_root):
    manifest_path = Path(mirror_root) / 'manifests' / 'mirror_manifest.json'
    if not manifest_path.exists():
        return manifest_path, {}
    payload = json.loads(manifest_path.read_text(encoding='utf-8'))
    declared_root = Path(payload.get('mirror_root', mirror_root))
    rows = {}
    for row in payload.get('artifacts', []):
        relative = row.get('relative_path')
        if not relative and row.get('mirror'):
            mirror_path = Path(row['mirror'])
            try:
                relative = mirror_path.relative_to(declared_root).as_posix()
            except ValueError:
                relative = mirror_path.name
        if relative:
            rows[Path(relative).as_posix()] = row
    return manifest_path, rows


def restore_required_revision_artifacts(required_rel_paths):
    import shutil

    destination_root = REPO_DIR / 'reports' / 'revision'
    manifest_path, rows = _mirror_manifest_rows(MIRROR_REVISION_ROOT)
    restored, mirror_path_copied, fallback_copied, reused, unresolved = [], [], [], [], []
    for rel in required_rel_paths:
        rel = Path(rel).as_posix()
        destination = destination_root / rel
        row = rows.get(rel)
        needs_copy = not destination.exists()
        if row and destination.exists() and _sha256_local(destination) != row.get('sha256'):
            needs_copy = True
        if not needs_copy:
            reused.append(rel)
            continue

        source = MIRROR_REVISION_ROOT / rel
        copied = False
        if row and source.exists():
            expected_size = int(row.get('size_bytes', -1))
            if expected_size >= 0 and source.stat().st_size != expected_size:
                raise RuntimeError(f'Mirror size mismatch for required artifact: {rel}')
            if _sha256_local(source) != row.get('sha256'):
                raise RuntimeError(f'Mirror checksum mismatch for required artifact: {rel}')
            destination.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(source, destination)
            if _sha256_local(destination) != row.get('sha256'):
                raise RuntimeError(f'Checksum mismatch after restoring required artifact: {rel}')
            restored.append(rel)
            copied = True

        if not copied and source.exists():
            destination.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(source, destination)
            mirror_path_copied.append(rel)
            copied = True
            print(f'WARNING: copied {rel} from mirror path without a manifest row; downstream SHA/protocol checks still apply.')

        fallback_source = DRIVE_REPO_REVISION_ROOT / rel
        if not copied and fallback_source.exists():
            destination.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(fallback_source, destination)
            fallback_copied.append(rel)
            copied = True
            print(f'WARNING: copied {rel} from Drive repo fallback because verified mirror copy was unavailable.')

        if not copied:
            unresolved.append(rel)

    print(
        'Required revision artifact restore: '
        f'restored={len(restored)} mirror_path={len(mirror_path_copied)} fallback={len(fallback_copied)} reused={len(reused)} unresolved={len(unresolved)}'
    )
    if restored:
        print('Restored required artifacts from verified mirror:')
        for rel in restored:
            print(' -', rel)
    if mirror_path_copied:
        print('Copied required artifacts from mirror path without manifest rows; downstream SHA/protocol checks still apply:')
        for rel in mirror_path_copied:
            print(' -', rel)
    if fallback_copied:
        print('Fallback-copied required artifacts from Drive repo; downstream SHA/protocol checks still apply:')
        for rel in fallback_copied:
            print(' -', rel)
    if unresolved:
        print('WARNING: required artifacts remain missing after mirror/fallback restore. Contract cell will fail if unresolved:')
        for rel in unresolved:
            print(' -', rel)


if MIRROR_REVISION_ROOT.exists():
    restore_result = _run_setup(
        f'python -u scripts/revision/artifact_mirror.py restore --mirror-root "{MIRROR_REVISION_ROOT}" --replace-mismatched',
        cwd=REPO_DIR,
        check=False,
    )
    if restore_result.returncode:
        print('WARNING: full checksum-verified mirror restore failed; trying selective restore for required inputs only.')
    restore_required_revision_artifacts(REQUIRED_REVISION_ARTIFACTS_FOR_05)
else:
    print('Mirror not found; trying Drive repo fallback for required artifacts:', MIRROR_REVISION_ROOT)
    restore_required_revision_artifacts(REQUIRED_REVISION_ARTIFACTS_FOR_05)

required_code = [
    'scripts/revision/09_hrv_domain_analysis.py',
    'notebooks/05_hrv_domain_and_robustness.ipynb',
    'docs/revision_plan/experiment_registry.csv',
    'docs/revision_plan/task_board.csv',
]
missing_code = []
for item in required_code:
    file_path = REPO_DIR / item
    status = 'FOUND' if file_path.exists() else 'MISSING'
    size = file_path.stat().st_size if file_path.exists() else None
    print(f'{item}: {status} size={size}')
    if not file_path.exists():
        missing_code.append(item)
if missing_code:
    raise FileNotFoundError('Missing required code files after setup: ' + '; '.join(missing_code))

cache_status = {
    'clean_raw_cache': DRIVE_ROOT / 'ecg_data_27c_subject.npz',
    'raw_minirocket_cache': DRIVE_ROOT / 'rocket_raw_N44186_C12_L5000_K10000_S42.npz',
    'hrv36_cache': DRIVE_ROOT / 'hrv36_N44186_C12_L5000.npz',
    'fold_pca_cache_dir': DRIVE_ROOT / 'revision_feature_cache',
    'model_dir_drive': DRIVE_ROOT / 'model',
}
print('cache policy:')
print('  ECG_RAMBA_USE_CLEAN_CACHE =', os.environ.get('ECG_RAMBA_USE_CLEAN_CACHE'))
print('  ECG_RAMBA_SAVE_CLEAN_CACHE=', os.environ.get('ECG_RAMBA_SAVE_CLEAN_CACHE'))
print('  ECG_RAMBA_EXTRACT_DIR     =', os.environ.get('ECG_RAMBA_EXTRACT_DIR'))
print('cache status:')
for name, cache_path in cache_status.items():
    if cache_path.is_dir():
        npz_count = len(list(cache_path.glob('*.npz')))
        pt_count = len(list(cache_path.glob('*.pt')))
        print(f'  {name}: exists=True npz_count={npz_count} pt_count={pt_count} path={cache_path}')
    else:
        print(f'  {name}: exists={cache_path.exists()} size={cache_path.stat().st_size if cache_path.exists() else None} path={cache_path}')

def run(cmd, check=True, log_path=None, tail_lines=160, cwd=None):
    print(f'$ {cmd}', flush=True)
    run_cwd = Path(cwd) if cwd else Path.cwd()
    if log_path is None:
        log_dir = run_cwd / 'reports' / 'revision' / 'logs'
        log_dir.mkdir(parents=True, exist_ok=True)
        stamp = time.strftime('%Y%m%d_%H%M%S')
        log_path = log_dir / f'notebook_command_{stamp}.log'
    else:
        log_path = Path(log_path)
        if not log_path.is_absolute():
            log_path = run_cwd / log_path
        log_path.parent.mkdir(parents=True, exist_ok=True)

    with log_path.open('w', encoding='utf-8') as log_file:
        proc = subprocess.Popen(
            cmd,
            shell=True,
            cwd=str(run_cwd),
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            encoding='utf-8',
            errors='replace',
            bufsize=1,
        )
        assert proc.stdout is not None
        for line in proc.stdout:
            print(line, end='')
            log_file.write(line)
            log_file.flush()
        return_code = proc.wait()

    print(f'Command log: {log_path}')
    if check and return_code:
        log_lines = log_path.read_text(encoding='utf-8', errors='replace').splitlines()
        print(f'Command failed with exit code {return_code}. Last {min(tail_lines, len(log_lines))} log lines:')
        for line in log_lines[-tail_lines:]:
            print(line)
        raise subprocess.CalledProcessError(return_code, cmd)
    return subprocess.CompletedProcess(cmd, return_code)


def run_script_if_exists(script_path, command):
    script = Path(script_path)
    if script.exists():
        run(command)
    else:
        print(f'SKIP: {script_path} is not implemented yet.')
        print(f'Planned command: {command}')

print('Setup ready. Continue with Scope And Contracts cell.')


## Scope And Contracts

This notebook now executes the implemented HRV-only and HRV-domain analysis runner. It still blocks robustness stress-test claims because perturbation inference is a separate heavy runner that has not been implemented in this branch. The HRV runner does not load neural checkpoints, perturb signals, or run full-model inference.


In [ ]:
from datetime import datetime, timezone
import math
import numpy as np
import pandas as pd
from IPython.display import display

from scripts.revision.common import git_commit, save_json, sha256_file

RUN_HRV_DOMAIN_ANALYSIS = True
RUN_ROBUSTNESS_STRESS = False

revision_root = Path('reports/revision')
OOF_STEM = 'oof_final_ema'
freeze_path = revision_root / 'manifests' / f'{OOF_STEM}_freeze_manifest.json'
oof_prediction_path = revision_root / 'predictions' / f'{OOF_STEM}_predictions.npz'
calibration_path = revision_root / 'metrics' / f'calibration_ci_{OOF_STEM}_predictions.json'
baseline_summary_path = revision_root / 'metrics' / 'baseline_summary.csv'
component_summary_path = revision_root / 'metrics' / 'component_check_summary.json'
audit_path = revision_root / 'audit_protocol.json'
a0_status_path = revision_root / 'a0_resolution_status.json'
hrv_schema_path = revision_root / 'hrv36_schema.csv'

required_inputs = {
    'oof_final_ema_freeze_manifest': freeze_path,
    'oof_final_ema_predictions': oof_prediction_path,
    'calibration_ci': calibration_path,
    'baseline_summary': baseline_summary_path,
    'component_check_summary': component_summary_path,
    'audit_protocol': audit_path,
    'a0_resolution_status': a0_status_path,
    'hrv36_schema': hrv_schema_path,
}
for name, path in required_inputs.items():
    print(f'{name:26s}: exists={path.exists()} size={path.stat().st_size if path.exists() else None} path={path}')
missing = [str(path) for path in required_inputs.values() if not path.exists()]
if missing and 'restore_required_revision_artifacts' in globals():
    print('Required inputs missing before contract validation; retrying selective artifact restore...')
    restore_required_revision_artifacts(REQUIRED_REVISION_ARTIFACTS_FOR_05)
    missing = [str(path) for path in required_inputs.values() if not path.exists()]
if missing:
    raise FileNotFoundError('Notebook 01/02/03/04 artifacts are required before notebook 05: ' + '; '.join(missing))

freeze = json.loads(freeze_path.read_text(encoding='utf-8'))
if freeze.get('status') != 'frozen' or freeze.get('manuscript_ready') is not True:
    raise ValueError('final_ema OOF freeze manifest must be status=frozen and manuscript_ready=true before HRV/robustness review.')
if freeze.get('checkpoint_kind') != 'final_ema':
    raise ValueError('Notebook 05 requires checkpoint_kind=final_ema in the freeze manifest.')
frozen_artifacts = {row['path']: row for row in freeze.get('artifacts', [])}
relative_oof_predictions = oof_prediction_path.as_posix()
if relative_oof_predictions not in frozen_artifacts:
    raise ValueError(f'Freeze manifest does not include {relative_oof_predictions}')
oof_prediction_sha = sha256_file(oof_prediction_path)
if oof_prediction_sha != frozen_artifacts[relative_oof_predictions]['sha256']:
    raise RuntimeError('OOF prediction SHA256 changed after final_ema freeze.')

baseline_df_for_contract = pd.read_csv(baseline_summary_path)
full_rows = baseline_df_for_contract[baseline_df_for_contract['baseline_name'] == 'Full ECG-RAMBA frozen OOF']
if full_rows.empty:
    raise RuntimeError('baseline_summary.csv lacks the Full ECG-RAMBA frozen OOF row from Notebook 04.')
if set(full_rows['evidence_path'].astype(str)) != {str(calibration_path)}:
    raise RuntimeError('baseline_summary.csv full-model row is not tied to the canonical final_ema calibration artifact.')
component_summary = json.loads(component_summary_path.read_text(encoding='utf-8'))
if component_summary.get('source_freeze_manifest') != str(freeze_path):
    raise RuntimeError('component_check_summary.json is not tied to the canonical final_ema freeze manifest.')
if component_summary.get('source_calibration') != str(calibration_path):
    raise RuntimeError('component_check_summary.json is not tied to the canonical final_ema calibration artifact.')

input_contract = {
    'created_utc': datetime.now(timezone.utc).isoformat(),
    'git_commit': git_commit(),
    'inputs': {
        name: {'path': str(path), 'sha256': sha256_file(path), 'size_bytes': path.stat().st_size}
        for name, path in required_inputs.items()
    },
    'freeze_status': freeze.get('status'),
    'freeze_manuscript_ready': freeze.get('manuscript_ready'),
    'freeze_checkpoint_kind': freeze.get('checkpoint_kind'),
    'oof_predictions_sha256': oof_prediction_sha,
    'allowed_execution': {
        'hrv_only_feature_baseline_training': RUN_HRV_DOMAIN_ANALYSIS,
        'hrv_domain_classifier_training': RUN_HRV_DOMAIN_ANALYSIS,
        'robustness_stress_runner': RUN_ROBUSTNESS_STRESS,
        'full_model_training': False,
        'full_model_inference': False,
        'signal_perturbation': False,
    },
}
save_json(revision_root / 'manifests' / 'hrv_robustness_input_contract.json', input_contract)
print('Input contract validated and written:', revision_root / 'manifests' / 'hrv_robustness_input_contract.json')


## Revision Plan Alignment

This cell binds notebook 05 to HRV-domain and robustness tasks in the tracked revision plan.


In [ ]:
registry = pd.read_csv('docs/revision_plan/experiment_registry.csv')
claims = pd.read_csv('docs/revision_plan/claim_evidence_map.csv')
tasks = pd.read_csv('docs/revision_plan/task_board.csv')

relevant_experiments = registry[registry['experiment_id'].isin(['EXP_HRV', 'EXP_ROBUST'])]
relevant_claims = claims[claims['claim_id'].isin(['C03'])]
relevant_tasks = tasks[tasks['id'].isin(['A6', 'B2'])]

display(relevant_experiments)
display(relevant_claims)
display(relevant_tasks)

plan_alignment = {
    'created_utc': datetime.now(timezone.utc).isoformat(),
    'git_commit': git_commit(),
    'experiments': relevant_experiments.to_dict(orient='records'),
    'claims': relevant_claims.to_dict(orient='records'),
    'tasks': relevant_tasks.to_dict(orient='records'),
}
save_json(revision_root / 'manifests' / 'hrv_robustness_plan_alignment.json', plan_alignment)
print('Wrote:', revision_root / 'manifests' / 'hrv_robustness_plan_alignment.json')


## HRV Domain Evidence

This cell runs the implemented HRV-only baseline and HRV-domain classifier. The HRV-only baseline uses HRV36 features with the same frozen Chapman OOF folds. The domain classifier uses bounded HRV36 feature caches only; it does not open raw signals or run the neural model.


In [ ]:
hrv_schema = pd.read_csv(hrv_schema_path)
print('HRV schema rows:', len(hrv_schema))
display(hrv_schema.head(10))

fingerprinted_hrv_caches = sorted(
    DRIVE_ROOT.glob('hrv36_N44186_C12_L5000_R*.npz'),
    key=lambda path: path.stat().st_mtime,
    reverse=True,
)
if not fingerprinted_hrv_caches:
    raise FileNotFoundError(
        'Missing record-fingerprinted Chapman HRV cache hrv36_N44186_C12_L5000_R*.npz on Drive. '
        'Run Notebook 02a/train cache generation first; do not use the legacy shape-only HRV cache for manuscript evidence.'
    )
chapman_hrv_cache = fingerprinted_hrv_caches[0]
chapman_hrv_cache_sha = sha256_file(chapman_hrv_cache)
print('Using fingerprinted Chapman HRV cache:', chapman_hrv_cache)
print('Chapman HRV cache sha256:', chapman_hrv_cache_sha)

hrv_command = (
    'python -u scripts/revision/09_hrv_domain_analysis.py '
    f'--oof-predictions {oof_prediction_path} '
    f'--freeze-manifest {freeze_path} --expected-checkpoint-kind final_ema '
    f'--chapman-hrv-cache "{chapman_hrv_cache}" '
    '--threshold 0.5 --n-bins 15 --n-boot 1000 --domain-max-per-domain 2000'
)
if RUN_HRV_DOMAIN_ANALYSIS:
    run(hrv_command, log_path='reports/revision/logs/hrv_domain_analysis.log')
else:
    raise RuntimeError('RUN_HRV_DOMAIN_ANALYSIS is False; HRV evidence will remain blocked.')

hrv_required_outputs = [
    revision_root / 'predictions' / 'hrv_only_oof_predictions.npz',
    revision_root / 'metrics' / 'hrv_only_baseline_summary.json',
    revision_root / 'metrics' / 'hrv_domain_classifier_summary.json',
    revision_root / 'metrics' / 'hrv_domain_summary.csv',
    revision_root / 'tables' / 'table_hrv_only_class_metrics.csv',
    revision_root / 'tables' / 'table_hrv_domain_status.csv',
    revision_root / 'manifests' / 'hrv_domain_analysis_manifest.json',
]
hrv_optional_outputs = [
    revision_root / 'predictions' / 'hrv_domain_oof_predictions.npz',
    revision_root / 'tables' / 'table_hrv_domain_classifier_confusion.csv',
    revision_root / 'tables' / 'table_hrv_domain_classifier_fold_summary.csv',
]
for output in hrv_required_outputs:
    print(output, 'required exists=', output.exists(), 'size=', output.stat().st_size if output.exists() else None)
missing_hrv = [str(output) for output in hrv_required_outputs if not output.exists()]
if missing_hrv:
    raise FileNotFoundError('Missing required HRV analysis outputs: ' + '; '.join(missing_hrv))
for output in hrv_optional_outputs:
    print(output, 'optional exists=', output.exists(), 'size=', output.stat().st_size if output.exists() else None)

hrv_df = pd.read_csv(revision_root / 'metrics' / 'hrv_domain_summary.csv')
hrv_baseline_summary = json.loads((revision_root / 'metrics' / 'hrv_only_baseline_summary.json').read_text(encoding='utf-8'))
hrv_domain_summary = json.loads((revision_root / 'metrics' / 'hrv_domain_classifier_summary.json').read_text(encoding='utf-8'))
hrv_load_info = hrv_baseline_summary.get('load_info', {})
if hrv_load_info.get('oof_predictions_sha256') != oof_prediction_sha:
    raise RuntimeError('HRV-only summary was not generated against the canonical final_ema OOF predictions.')
if hrv_load_info.get('chapman_hrv_cache_kind') != 'record_fingerprinted':
    raise RuntimeError('HRV-only summary did not use a record-fingerprinted Chapman HRV cache.')
if hrv_load_info.get('chapman_hrv_cache_sha256') != chapman_hrv_cache_sha:
    raise RuntimeError('HRV-only summary cache SHA does not match the selected fingerprinted HRV cache.')
if hrv_load_info.get('freeze_contract', {}).get('checkpoint_kind') != 'final_ema':
    raise RuntimeError('HRV-only summary is not tied to final_ema freeze contract.')

print('HRV-only metrics:', json.dumps(hrv_baseline_summary['metrics'], indent=2, sort_keys=True))
print('HRV domain status:', hrv_domain_summary.get('status', 'complete'))
print('HRV domain metrics:', json.dumps(hrv_domain_summary.get('metrics') or {}, indent=2, sort_keys=True))
print('HRV domain interpretation:', hrv_domain_summary.get('interpretation'))
if hrv_domain_summary.get('blocker'):
    print('HRV domain blocker:', hrv_domain_summary.get('blocker'))
display(hrv_df)


## Robustness Stress-Test Ledger

The revised plan requires perturbation tests. Since no runner exists in this branch, the notebook writes a blocked but reusable robustness table instead of fabricating metrics.


In [ ]:
robustness_rows = []
for analysis_name, perturbation in [
    ('SNR 20 dB noise', 'additive_noise_snr_20db'),
    ('SNR 10 dB noise', 'additive_noise_snr_10db'),
    ('SNR 5 dB noise', 'additive_noise_snr_5db'),
    ('Random 3-lead dropout', 'random_3_lead_dropout'),
    ('V1-V6 dropout', 'precordial_v1_v6_dropout'),
    ('500Hz to 250Hz sampling shift', 'sampling_rate_500_to_250hz'),
]:
    robustness_rows.append({
        'analysis_name': analysis_name,
        'perturbation': perturbation,
        'dataset': 'chapman_oof',
        'status': 'blocked_runner_tbd',
        'clean_metric_reference': str(calibration_path),
        'perturbed_metric_value': math.nan,
        'absolute_delta': math.nan,
        'relative_delta': math.nan,
        'evidence_path': '',
        'blocker': 'No implemented robustness stress-test runner in scripts/revision.',
        'safe_wording': 'Do not claim robustness under this perturbation until the runner is implemented and artifacts are frozen.',
    })

robustness_df = pd.DataFrame(robustness_rows)
robustness_csv = revision_root / 'metrics' / 'robustness_summary.csv'
robustness_table = revision_root / 'tables' / 'table_robustness.csv'
robustness_df.to_csv(robustness_csv, index=False)
robustness_df.to_csv(robustness_table, index=False)
print('Wrote:', robustness_csv)
print('Wrote:', robustness_table)
display(robustness_df)


## Claim Evidence Summary

This JSON is the rebuttal-facing interpretation boundary for HRV and robustness claims.


In [ ]:
hrv_summary_csv = revision_root / 'metrics' / 'hrv_domain_summary.csv'
hrv_df = pd.read_csv(hrv_summary_csv)
hrv_statuses = dict(zip(hrv_df['analysis_name'], hrv_df['status']))
hrv_baseline_summary_path = revision_root / 'metrics' / 'hrv_only_baseline_summary.json'
hrv_domain_classifier_summary_path = revision_root / 'metrics' / 'hrv_domain_classifier_summary.json'
hrv_baseline_summary = json.loads(hrv_baseline_summary_path.read_text(encoding='utf-8'))
hrv_domain_classifier_summary = json.loads(hrv_domain_classifier_summary_path.read_text(encoding='utf-8'))
hrv_only_complete = hrv_statuses.get('HRV-only baseline') == 'complete'
hrv_domain_complete = hrv_statuses.get('HRV domain classifier') == 'complete'
if hrv_only_complete and hrv_domain_complete:
    hrv_status = 'hrv_only_and_domain_classifier_complete_duration_noise_blocked'
elif hrv_only_complete:
    hrv_status = 'hrv_only_complete_domain_classifier_blocked_or_missing_duration_noise_blocked'
else:
    hrv_status = 'blocked_or_incomplete'

summary = {
    'created_utc': datetime.now(timezone.utc).isoformat(),
    'git_commit': git_commit(),
    'source_freeze_manifest': str(freeze_path),
    'source_calibration': str(calibration_path),
    'hrv_schema': {'path': str(hrv_schema_path), 'sha256': sha256_file(hrv_schema_path)},
    'hrv_status': hrv_status,
    'robustness_status': 'blocked_runner_tbd',
    'hrv_metrics': {
        'hrv_only_roc_auc_macro': hrv_baseline_summary['metrics'].get('roc_auc_macro'),
        'hrv_only_pr_auc_macro': hrv_baseline_summary['metrics'].get('pr_auc_macro'),
        'hrv_only_f1_macro': hrv_baseline_summary['metrics'].get('f1_macro'),
        'domain_status': hrv_domain_classifier_summary.get('status', 'complete' if hrv_domain_complete else 'blocked_or_missing'),
        'domain_roc_auc_ovr_macro': (hrv_domain_classifier_summary.get('metrics') or {}).get('domain_roc_auc_ovr_macro'),
        'domain_balanced_accuracy': (hrv_domain_classifier_summary.get('metrics') or {}).get('balanced_accuracy'),
        'domain_interpretation': hrv_domain_classifier_summary.get('interpretation'),
        'domain_blocker': hrv_domain_classifier_summary.get('blocker'),
    },
    'claim_boundaries': [
        {
            'claim_id': 'C03',
            'claim_topic': 'HRV provides complementary rhythm descriptors',
            'current_status': 'partially_supported_by_hrv_only_only' if hrv_only_complete and not hrv_domain_complete else ('partially_supported_by_hrv_only_and_domain_classifier' if hrv_only_complete else 'not_supported_by_current_artifacts'),
            'safe_wording': 'HRV-only can be reported as a separate feature baseline. Domain-sensitivity evidence requires external HRV36 caches if the domain classifier is blocked. Duration/noise HRV sensitivity remains blocked.',
        },
        {
            'claim_id': 'A6/R2C3',
            'claim_topic': 'minimum robustness stress tests',
            'current_status': 'not_supported_by_current_artifacts',
            'safe_wording': 'Robustness tests must be implemented and reported as absolute/relative degradation before any robustness claim.',
        },
    ],
}
summary_path = revision_root / 'metrics' / 'hrv_robustness_component_check_summary.json'
save_json(summary_path, summary)
print('Wrote:', summary_path)
print(json.dumps(summary['hrv_metrics'], indent=2, sort_keys=True))


## Remaining Runner Guard

This cell records the implemented HRV runner and keeps robustness stress testing fail-closed until a dedicated perturbation inference runner exists.


In [ ]:
planned = {
    'HRV-only and domain analysis': 'python scripts/revision/09_hrv_domain_analysis.py --oof-predictions reports/revision/predictions/oof_final_ema_predictions.npz --freeze-manifest reports/revision/manifests/oof_final_ema_freeze_manifest.json --expected-checkpoint-kind final_ema --chapman-hrv-cache /content/drive/MyDrive/ECG-Ramba/hrv36_N44186_C12_L5000_R*.npz',
    'Duration/noise HRV sensitivity': 'python scripts/revision/TBD_hrv_duration_noise_sensitivity.py',
    'Robustness stress tests': 'python scripts/revision/TBD_robustness_stress.py',
}

for name, command in planned.items():
    script = Path(command.split()[1])
    print(f'{name:36s}: implemented={script.exists()} planned_command={command}')

if RUN_ROBUSTNESS_STRESS:
    missing = [name for name, command in planned.items() if not Path(command.split()[1]).exists()]
    if missing:
        raise RuntimeError('Remaining HRV/robustness runners are not implemented yet: ' + ', '.join(missing))
else:
    print('Robustness stress execution disabled. Current notebook keeps robustness claims blocked until a perturbation inference runner exists.')


## Inventory And Stable Mirror

Freeze the revised artifact inventory and publish all reusable outputs to the Drive mirror.


In [ ]:
import inspect
if 'log_path' not in inspect.signature(run).parameters:
    raise RuntimeError('Notebook command helper run() was overwritten. Rerun the Setup cell before this inventory/mirror cell.')

run(
    'python -u scripts/revision/05_artifact_inventory.py',
    log_path='reports/revision/logs/hrv_robustness_artifact_inventory.log',
)
mirror_root = DRIVE_ROOT / 'revision_artifacts' / 'reports' / 'revision'
run(
    f'python -u scripts/revision/artifact_mirror.py publish --mirror-root "{mirror_root}"',
    log_path='reports/revision/logs/hrv_robustness_mirror_publish.log',
)


## Output Summary


In [ ]:
expected_outputs = [
    revision_root / 'manifests' / 'hrv_robustness_input_contract.json',
    revision_root / 'manifests' / 'hrv_robustness_plan_alignment.json',
    revision_root / 'manifests' / 'hrv_domain_analysis_manifest.json',
    revision_root / 'predictions' / 'hrv_only_oof_predictions.npz',
    revision_root / 'metrics' / 'hrv_only_baseline_summary.json',
    revision_root / 'metrics' / 'hrv_domain_classifier_summary.json',
    revision_root / 'metrics' / 'hrv_domain_summary.csv',
    revision_root / 'tables' / 'table_hrv_domain_status.csv',
    revision_root / 'tables' / 'table_hrv_only_class_metrics.csv',
    revision_root / 'metrics' / 'robustness_summary.csv',
    revision_root / 'tables' / 'table_robustness.csv',
    revision_root / 'metrics' / 'hrv_robustness_component_check_summary.json',
    revision_root / 'manifests' / 'artifacts_manifest.json',
    revision_root / 'manifests' / 'artifacts_manifest.csv',
]
optional_outputs = [
    revision_root / 'predictions' / 'hrv_domain_oof_predictions.npz',
    revision_root / 'tables' / 'table_hrv_domain_classifier_confusion.csv',
    revision_root / 'tables' / 'table_hrv_domain_classifier_fold_summary.csv',
]
for path in expected_outputs:
    print(path, 'required exists=', path.exists(), 'size=', path.stat().st_size if path.exists() else None)
for path in optional_outputs:
    print(path, 'optional exists=', path.exists(), 'size=', path.stat().st_size if path.exists() else None)

print()
print('Notebook 05 complete. HRV-only evidence is implemented; HRV domain evidence is complete only when external HRV36 caches exist. Robustness claims remain blocked until perturbation inference artifacts exist.')
